# N13 — Safety Car Probability: EDA & Labeling

The goal of this notebook is to build the labeled dataset for the safety car probability
model (N14). We work at the level of **race laps**: one row per (race, lap_number),
aggregating the observable state of the field at that point in the race.

For each lap we determine whether a Safety Car or Virtual Safety Car deployment
occurred within the next 5 laps — defining the binary `sc_within_5_laps` label.

The output is a Parquet file with race-state features and a binary SC label,
ready for Random Forest training in N14.

> **Dataset note:** unlike the overtake dataset (~28k pairs), the SC dataset is
> significantly smaller — ~3,500 rows (58 races × ~60 laps). Class imbalance
> and validation strategy are designed accordingly.

**Data sources:**
- `session.laps` — lap times, tyre data, positions, retirements (2023–2025)
- `session.track_status` — timestamped track status changes at second-level resolution;
  used to derive yellow flag counts and escalation patterns per lap window
- `session.race_control_messages` — timestamped FIA messages (incidents, yellow sectors,
  SC/VSC announcements); captures the precursor signals that precede SC deployments
- `data/processed/circuit_clustering/` — circuit cluster assignments

**Exports:**
- EDA plots → `notebooks/strategy/sc_probability/outputs/`
- Labeled dataset → `data/processed/sc_labeled/`


---

## Step 0 - Setup and Imports 

In [ ]:
# ── Step 0 — Setup ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
import pathlib
import json
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

# ── Repo root ─────────────────────────────────────────────────────────────────
repo_root = pathlib.Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUTS    = repo_root / "notebooks" / "strategy" / "sc_probability" / "outputs"
PROCESSED  = repo_root / "data" / "processed" / "sc_labeled"
CACHE      = repo_root / "data" / "cache" / "fastf1"
CLUSTERING = repo_root / "data" / "processed" / "circuit_clustering"

OUTPUTS.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

import logging
logging.getLogger("fastf1").setLevel(logging.WARNING)
fastf1.Cache.enable_cache(str(CACHE))


In [ ]:
print(f"Repo root  : {repo_root}")
print(f"Outputs    : {OUTPUTS}")
print(f"Processed  : {PROCESSED}")
print(f"Clustering : {CLUSTERING}")
print(f"FF1 cache  : {CACHE}")

---

## Step 1 — Schedule & Circuit Clusters

Load the race calendar for 2023–2025 and attach the circuit cluster label to each
event. The cluster (0–3) encodes the circuit type (power, high-downforce, balanced,
street) and will be used as a categorical feature in N14.

**Circuit cluster source:** `data/processed/circuit_clustering/circuit_clusters_k4.parquet`
— produced in the circuit clustering notebook and reused across all strategy models.


In [ ]:
# ── Step 1 — Schedule & circuit clusters ──────────────────────────────────────
SEASONS = [2023, 2024, 2025]


def load_circuit_clusters(clustering_path: pathlib.Path) -> dict:
    """Load GP_Name → Cluster mapping from the k4 parquet file."""
    df = pd.read_parquet(clustering_path / "circuit_clusters_k4.parquet")
    return dict(zip(df["GP_Name"], df["Cluster"]))


def resolve_cluster(location: str, cluster_map: dict) -> int:
    """Fuzzy-match a FastF1 Location string to a cluster id. Returns -1 if unknown."""
    for key, cluster in cluster_map.items():
        if key.lower() in location.lower() or location.lower() in key.lower():
            return cluster
    return -1


def load_schedule(seasons: list, cluster_map: dict) -> pd.DataFrame:
    """Fetch race calendars for each season, filter to Race rounds, attach cluster."""
    rows = []
    for year in seasons:
        sched = fastf1.get_event_schedule(year, include_testing=False)
        races = sched[
            sched["EventFormat"].isin(["conventional", "sprint_shootout", "sprint"]) &
            (sched["Session5"] == "Race")
        ].copy()
        races["year"] = year
        rows.append(races[["year", "RoundNumber", "EventName", "Location", "Country"]])
    schedule = pd.concat(rows, ignore_index=True)
    schedule["circuit_cluster"] = schedule["Location"].apply(
        lambda loc: resolve_cluster(loc, cluster_map)
    )
    return schedule


# ── Run ───────────────────────────────────────────────────────────────────────
cluster_map = load_circuit_clusters(CLUSTERING)
schedule    = load_schedule(SEASONS, cluster_map)

unknown = schedule[schedule["circuit_cluster"] == -1]
print(f"Circuit clusters loaded : {len(cluster_map)} circuits")
print(f"Schedule loaded         : {len(schedule)} races across {SEASONS}")
print(f"Unknown clusters        : {len(unknown)}")
if len(unknown):
    print(unknown[["year", "EventName", "Location"]].to_string(index=False))
print(f"\nCluster distribution:\n{schedule['circuit_cluster'].value_counts().sort_index()}")


---

## Step 2 — Load Race Laps, Track Status & Race Control Messages

For each of the 58 races (2023–2025) we load three data sources per session:

- **`session.laps`** — one row per driver per lap; aggregated to race-lap level for
  the base features (lap times, tyre life, retirements).
- **`session.track_status`** — timestamped DataFrame recording every track status
  change during the race at second-level resolution. Unlike the per-lap
  `TrackStatus` field on laps, this gives us the *exact moment* a yellow or SC
  was called, allowing us to count how many status changes occurred within each
  lap window.
- **`session.race_control_messages`** — timestamped FIA messages including incident
  reports, yellow flag sector notifications, and SC/VSC announcements. These are
  the observable precursor signals that precede most SC deployments by 1–2 laps.

`track_status` and `race_control_messages` are stored as per-race dicts keyed by
`"{year}_{round}"` and used in Step 3 to engineer the richer lap-window features.

### TrackStatus field

`TrackStatus` is a **concatenated string of single-digit FIA codes** — a lap that
crosses a status boundary carries both codes (e.g. `'21'` for yellow then clear,
`'46'` for SC entering VSC). The full code table:

| Code | Status | Race relevance |
|------|--------|----------------|
| `'1'` | Track clear — green flag | Normal racing |
| `'2'` | Yellow flag (local caution) | Incident in sector |
| `'4'` | **Safety Car deployed** | Full-course caution |
| `'5'` | Red flag — session stopped | Rare; aborts the race |
| `'6'` | **Virtual Safety Car** | Speed-delta caution |
| `'7'` | VSC ending (transition) | Brief before returning to green |

`most_severe_status` parses each character independently and returns the single
highest-severity code: `'4'` (SC) > `'6'` (VSC) > `'5'` (red) > `'7'` (VSC ending)
> `'2'` (yellow) > `'1'` (green).


In [ ]:
# ── Step 2 — Load race laps, track status & race control messages ─────────────

STATUS_SEVERITY = {"1": 1, "2": 2, "5": 3, "7": 4, "6": 5, "4": 6}


def most_severe_status(ts_str: str) -> str:
    """Return the single character with the highest severity in a TrackStatus string."""
    if not isinstance(ts_str, str) or ts_str == "":
        return "1"
    chars = [c for c in ts_str if c in STATUS_SEVERITY]
    if not chars:
        return "1"
    return max(chars, key=lambda c: STATUS_SEVERITY[c])


# ── Driver-level feature helpers ──────────────────────────────────────────────

def compute_driver_level_features(laps: pd.DataFrame) -> dict:
    """
    Compute driver-level anomaly signals from per-driver lap data.

    Returns a dict of scalar values for one race lap:
      driver_anomaly_soft_count  — drivers with lap_time > 1.15x their rolling median
      driver_anomaly_hard_count  — drivers with lap_time > 1.30x their rolling median
      max_speed_drop_pct         — max speed trap drop vs 5-lap rolling avg
      tyre_age_high_risk_count   — drivers past compound cliff thresholds
      active_pitstop_count       — drivers pitting this lap
      outlap_drivers             — drivers on cold outlaps (TyreLife ≤ 2)

    All features are causally valid: computed from data already observed at the
    START of the current lap (rolling windows use only past laps via shift(1)).
    """
    # Filter to clean laps only (exclude pit in/out and SC laps for rolling medians)
    clean = laps[~laps["PitInTime"].notna() & ~laps["PitOutTime"].notna()].copy()

    # ── driver_anomaly_soft_count / hard_count ──────────────────────────────
    # Rolling median per driver over past 5 laps (min 2 periods), shifted by 1
    # to ensure we only use laps L-5..L-1, not the current lap.
    if not clean.empty and "LapTime" in clean.columns:
        lt_sec = clean["LapTime"].dt.total_seconds()
        rolling_med = (
            lt_sec.groupby(clean["Driver"])
            .transform(lambda s: s.shift(1).rolling(5, min_periods=2).median())
        )
        ratio = lt_sec / rolling_med
        soft_mask = ratio > 1.15
        hard_mask = ratio > 1.30
        anomaly_soft = int(soft_mask.sum())
        anomaly_hard = int(hard_mask.sum())
    else:
        anomaly_soft = 0
        anomaly_hard = 0

    # ── max_speed_drop_pct ──────────────────────────────────────────────────
    if "SpeedST" in laps.columns and not laps.empty:
        spd = laps["SpeedST"].dropna()
        if not spd.empty:
            rolling_spd_mean = (
                spd.groupby(laps.loc[spd.index, "Driver"])
                .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
            )
            drop_pct = ((rolling_spd_mean - spd) / rolling_spd_mean.clip(lower=1)).clip(lower=0)
            max_drop = float(drop_pct.max()) if not drop_pct.empty else 0.0
        else:
            max_drop = 0.0
    else:
        max_drop = 0.0

    # ── tyre_age_high_risk_count ─────────────────────────────────────────────
    CLIFF_THRESHOLDS = {"SOFT": 20, "MEDIUM": 35, "HARD": 50}
    if "TyreLife" in laps.columns and "Compound" in laps.columns:
        high_risk = 0
        for _, row in laps.iterrows():
            cmp = str(row.get("Compound", "")).upper()
            thresh = CLIFF_THRESHOLDS.get(cmp, 999)
            try:
                tl = float(row["TyreLife"])
            except (TypeError, ValueError):
                tl = 0.0
            if tl > thresh:
                high_risk += 1
    else:
        high_risk = 0

    # ── active_pitstop_count + outlap_drivers ────────────────────────────────
    pit_count   = int(laps["PitInTime"].notna().sum())  if "PitInTime"  in laps.columns else 0
    outlap_drv  = int((laps["TyreLife"] <= 2).sum())    if "TyreLife"   in laps.columns else 0

    return {
        "driver_anomaly_soft_count": anomaly_soft,
        "driver_anomaly_hard_count": anomaly_hard,
        "max_speed_drop_pct":        max_drop,
        "tyre_age_high_risk_count":  high_risk,
        "active_pitstop_count":      pit_count,
        "outlap_drivers":            outlap_drv,
    }


def compute_weather_features(session) -> dict:
    """
    Extract weather scalars for the race, averaged over the session.

    Bug fix v3: use forward-fill within the weather DataFrame before falling
    back to the global median, so a sensor dropout mid-race doesn't produce
    a constant fill value for the whole race.

    Returns track_temp, air_temp, humidity, track_temp_delta (last - first).
    Also returns weather_available = 1 if session.weather had non-trivial variance.
    """
    default = {"track_temp": np.nan, "air_temp": np.nan,
               "humidity": np.nan, "track_temp_delta": 0.0,
               "weather_available": 0}
    try:
        wx = session.weather_data
        if wx is None or wx.empty:
            return default
        # forward-fill first within the race, then backward-fill for leading NaN
        wx = wx.ffill().bfill()
        track_temp       = float(wx["TrackTemp"].mean())   if "TrackTemp"   in wx.columns else np.nan
        air_temp         = float(wx["AirTemp"].mean())     if "AirTemp"     in wx.columns else np.nan
        humidity         = float(wx["Humidity"].mean())    if "Humidity"    in wx.columns else np.nan
        track_temp_delta = (
            float(wx["TrackTemp"].iloc[-1] - wx["TrackTemp"].iloc[0])
            if "TrackTemp" in wx.columns and len(wx) > 1 else 0.0
        )
        # available flag: variance > 0.01 in at least one channel
        var_ok = any(
            wx[c].var() > 0.01
            for c in ["TrackTemp", "AirTemp", "Humidity"]
            if c in wx.columns
        )
        return {
            "track_temp":       track_temp,
            "air_temp":         air_temp,
            "humidity":         humidity,
            "track_temp_delta": track_temp_delta,
            "weather_available": int(var_ok),
        }
    except Exception:
        return default


# ── Aggregate laps to race-lap level ─────────────────────────────────────────

def aggregate_laps(session_laps: pd.DataFrame, total_laps: int,
                   weather: dict) -> pd.DataFrame:
    """
    Aggregate per-driver laps into one row per (race, LapNumber).

    Feature additions v3:
      lap_time_cv       — std / mean (coefficient of variation; circuit-normalised)
      lap_time_trend_5  — mean of last 5 laps / mean of prev 5 laps (pace deterioration)
      n_drivers_delta   — change in active driver count vs previous lap
      status_change_direction — +1 escalating, -1 de-escalating, 0 unchanged

    All rolling/lag features are causally valid (shift(1) applied).
    """
    lap_groups = session_laps.groupby("LapNumber", sort=True)

    rows = []
    prev_n_drivers    = None
    prev_status_sev   = None
    lap_time_mean_hist = []  # running list of per-lap mean times (for trend_5)

    for lap_num, grp in lap_groups:
        lt = grp["LapTime"].dropna().dt.total_seconds()
        lt_mean = float(lt.mean()) if not lt.empty else np.nan
        lt_std  = float(lt.std())  if len(lt) > 1  else 0.0
        lt_min  = float(lt.min())  if not lt.empty else np.nan

        # Coefficient of variation
        lt_cv   = (lt_std / lt_mean) if (lt_mean and lt_mean > 0) else 0.0

        # n_drivers and n_drivers_delta
        n_drv   = int(grp["Driver"].nunique())
        n_delta = (n_drv - prev_n_drivers) if prev_n_drivers is not None else 0
        prev_n_drivers = n_drv

        # Tyre life
        tl = grp["TyreLife"].dropna()
        tl_mean = float(tl.mean()) if not tl.empty else np.nan
        tl_max  = float(tl.max())  if not tl.empty else np.nan

        # Track status (most severe for this lap)
        ts_raw  = grp["TrackStatus"].dropna().tolist()
        ts_str  = "".join(ts_raw) if ts_raw else "1"
        ts_code = most_severe_status(ts_str)

        # status_change_direction: +1 escalating, -1 de-escalating, 0 same/first
        cur_sev = STATUS_SEVERITY.get(ts_code, 1)
        if prev_status_sev is None:
            sc_dir = 0
        elif cur_sev > prev_status_sev:
            sc_dir = 1
        elif cur_sev < prev_status_sev:
            sc_dir = -1
        else:
            sc_dir = 0
        prev_status_sev = cur_sev

        # Driver-level anomaly signals (per-lap cross-driver aggregation)
        drv_feats = compute_driver_level_features(grp)

        # ── lap_time_trend_5 ─────────────────────────────────────────────────
        # Ratio of mean of last 5 laps to mean of previous 5 laps.
        # "last 5" = laps L-4 to L (inclusive of current lap in the historical list
        # after appending), "prev 5" = laps L-9 to L-5.
        # We append the current lap mean AFTER computing trend (causal).
        n_hist = len(lap_time_mean_hist)
        if n_hist >= 5:
            last5_vals = [v for v in lap_time_mean_hist[-5:] if not np.isnan(v)]
            prev5_vals = [v for v in lap_time_mean_hist[-10:-5] if not np.isnan(v)] if n_hist >= 10 else []
            last5_mean = float(np.mean(last5_vals)) if last5_vals else np.nan
            prev5_mean = float(np.mean(prev5_vals)) if prev5_vals else np.nan
            if prev5_mean and prev5_mean > 0 and not np.isnan(last5_mean):
                lt_trend5 = last5_mean / prev5_mean
            else:
                lt_trend5 = 1.0
        else:
            lt_trend5 = 1.0
        lap_time_mean_hist.append(lt_mean)  # append AFTER computing (causal)

        row = {
            "LapNumber":                  int(lap_num),
            "lap_time_mean":              lt_mean,
            "lap_time_std":               lt_std,
            "lap_time_min":               lt_min,
            "lap_time_cv":                round(lt_cv, 6),
            "lap_time_trend_5":           round(lt_trend5, 6),
            "n_drivers":                  n_drv,
            "n_drivers_delta":            n_delta,
            "tyre_life_mean":             tl_mean,
            "tyre_life_max":              tl_max,
            "track_status":               ts_code,
            "status_change_direction":    sc_dir,
            # driver-level anomaly features
            "driver_anomaly_soft_count":  drv_feats["driver_anomaly_soft_count"],
            "driver_anomaly_hard_count":  drv_feats["driver_anomaly_hard_count"],
            "max_speed_drop_pct":         drv_feats["max_speed_drop_pct"],
            "tyre_age_high_risk_count":   drv_feats["tyre_age_high_risk_count"],
            "active_pitstop_count":       drv_feats["active_pitstop_count"],
            "outlap_drivers":             drv_feats["outlap_drivers"],
            # weather
            "track_temp":                 weather.get("track_temp",       np.nan),
            "air_temp":                   weather.get("air_temp",         np.nan),
            "humidity":                   weather.get("humidity",         np.nan),
            "track_temp_delta":           weather.get("track_temp_delta", 0.0),
            "weather_available":          weather.get("weather_available", 0),
        }
        rows.append(row)

    agg = pd.DataFrame(rows)

    # lap_pct (fraction of race completed)
    if total_laps and total_laps > 0:
        agg["lap_pct"] = agg["LapNumber"] / total_laps
    else:
        agg["lap_pct"] = 0.0

    # is_lap1
    agg["is_lap1"] = (agg["LapNumber"] == agg["LapNumber"].min()).astype(int)

    # track_status_enc (ordinal encoding)
    STATUS_ENC = {"1": 0, "2": 1, "5": 2, "7": 3, "6": 4, "4": 5}
    agg["track_status_enc"] = agg["track_status"].map(STATUS_ENC).fillna(0).astype(int)

    return agg


# ── Single session loader ─────────────────────────────────────────────────────

def load_single_session(year: int, round_num: int, event_name: str,
                        circuit_cluster: int,
                        cache_path: pathlib.Path) -> tuple:
    """
    Load one race session and return (agg_df, ts_df, rcm_df).

    Returns (None, None, None) on failure.
    """
    try:
        session = fastf1.get_session(year, round_num, "Race")
        session.load(laps=True, weather=True, messages=True, telemetry=False)
    except Exception as e:
        print(f"  [WARN] {year} R{round_num} ({event_name}): load failed — {e}")
        return None, None, None

    laps = session.laps.copy()
    if laps.empty:
        print(f"  [WARN] {year} R{round_num} ({event_name}): empty laps")
        return None, None, None

    # Total laps for lap_pct
    total_laps = int(laps["LapNumber"].max()) if not laps.empty else 0

    # Weather
    weather = compute_weather_features(session)

    # Aggregate to race-lap level
    agg = aggregate_laps(laps, total_laps, weather)
    agg["year"]            = year
    agg["round"]           = round_num
    agg["event_name"]      = event_name
    agg["circuit_cluster"] = circuit_cluster
    agg["race_id"]         = f"{year}_{round_num}"

    # Track status timestamped (for Step 3 feature engineering)
    try:
        ts_df = session.track_status.copy()
        ts_df["race_id"] = f"{year}_{round_num}"
    except Exception:
        ts_df = pd.DataFrame()

    # Race control messages
    try:
        rcm_df = session.race_control_messages.copy()
        # Join lap numbers if not already present
        if "Lap" not in rcm_df.columns and not laps.empty:
            lap_times = (
                laps.groupby("LapNumber")["LapStartTime"]
                .min().reset_index()
                .sort_values("LapNumber")
            )
            lap_times["LapEnd"] = lap_times["LapStartTime"].shift(-1)
            def assign_lap(t):
                mask = (lap_times["LapStartTime"] <= t) & (lap_times["LapEnd"].isna() | (t < lap_times["LapEnd"]))
                matched = lap_times[mask]["LapNumber"]
                return int(matched.iloc[0]) if not matched.empty else np.nan
            if "Time" in rcm_df.columns:
                rcm_df["Lap"] = rcm_df["Time"].apply(assign_lap)
        rcm_df["race_id"] = f"{year}_{round_num}"
    except Exception:
        rcm_df = pd.DataFrame()

    return agg, ts_df, rcm_df


# ── Run — load all sessions ───────────────────────────────────────────────────

fastf1.Cache.enable_cache(str(CACHE))

raw_rows   = []
ts_store   = {}
rcm_store  = {}
fail_count = 0

for _, row in schedule.iterrows():
    yr         = int(row["year"])
    rnd        = int(row["RoundNumber"])
    evt        = str(row["EventName"])
    clust      = int(row["circuit_cluster"])
    race_id    = f"{yr}_{rnd}"

    agg, ts_df, rcm_df = load_single_session(yr, rnd, evt, clust, CACHE)
    if agg is None:
        fail_count += 1
        continue

    raw_rows.append(agg)
    ts_store[race_id]  = ts_df
    rcm_store[race_id] = rcm_df

raw = pd.concat(raw_rows, ignore_index=True)

print(f"Sessions loaded  : {len(raw_rows)} / {len(schedule)} (failures: {fail_count})")
print(f"Lap rows         : {len(raw):,}")
print(f"Columns          : {list(raw.columns)}")
print(f"\nWeather available: {raw['weather_available'].mean()*100:.1f}% of races have non-trivial weather variance")
print(f"Status direction dist:\n{raw['status_change_direction'].value_counts().sort_index()}")
print(f"\nDriver anomaly (soft>0): {(raw['driver_anomaly_soft_count']>0).sum()}")
print(f"Driver anomaly (hard>0): {(raw['driver_anomaly_hard_count']>0).sum()}")


**Step 2 results — 58 races loaded, 0 failures, 3,506 lap-rows.**

Track status breakdown across all laps:

| Status | Code | Count | % laps |
|--------|------|-------|--------|
| Green  | `1`  | 3,133 | 89.4%  |
| Yellow | `2`  |   140 |  4.0%  |
| SC     | `4`  |   173 |  4.9%  |
| VSC    | `6`  |    58 |  1.7%  |
| Red    | `5`  |     2 |  0.1%  |

SC + VSC combined: **231 caution laps (6.6%)** — this is the raw caution rate at lap
level, before the 5-lap forward window in Step 4 inflates the positive class.
The 140 yellow-flag laps are candidate precursors; whether they systematically precede
SC deployments is exactly what the EDA in Step 5 will confirm.

Two useful structural findings from the loaded dicts:
- `ts_store` carries a `Message` column alongside `Time`/`Status`, providing
  human-readable context for each status change (e.g. `"SAFETY CAR DEPLOYED"`).
- `rcm_store` includes a `Lap` column — direct integer join by lap number in Step 3,
  no timestamp interpolation needed.


---

## Step 3 — Feature Engineering

We enrich the base lap-level table with features derived from the two timestamped
sources loaded in Step 2. The goal is to capture the *observable race state* at
each lap — not just what is happening, but what has been building up.

**Driver-level anomaly features** (computed in Step 2 before aggregation) — v3 dual threshold:
- `driver_anomaly_soft_count` — count of drivers with lap time >1.15× their personal
  rolling 5-lap median, excluding pit/outlaps. Soft threshold catches slow-moving cars
  and minor incidents. (Replaces `driver_anomaly_count` which was near-zero with 1.5×.)
- `driver_anomaly_hard_count` — count of drivers with lap time >1.30× rolling median.
  Crash or major mechanical failure signal.
- `max_speed_drop_pct` — largest speed trap drop (vs rolling 5-lap avg) across any
  driver. Captures mechanical failures and tyre/floor damage signals.
- `tyre_age_high_risk_count` — drivers past compound cliff thresholds (SOFT>20 laps,
  MEDIUM>35, HARD>50). High tyre age → blowout and grip loss risk.
- `active_pitstop_count` — drivers pitting on this lap. Pit lane entry traffic.
- `outlap_drivers` — drivers on cold outlaps (TyreLife ≤ 2). Slow cars creating gaps.

**New pace features** (computed in `aggregate_laps`, v3):
- `lap_time_cv` — coefficient of variation (std/mean); normalises pace spread across circuits.
- `lap_time_trend_5` — mean of last 5 laps / mean of previous 5 laps. Captures pace
  deterioration. Values > 1.0 = slowing down. Default 1.0 before enough laps.
- `n_drivers_delta` — change in active driver count vs previous lap. Negative = retirement.

**From `raw["track_status"]`** (per-lap most-severe code) — v3 status features:
- `yellow_escalation_count` — laps in prev 3 where severity *increased* (replaces
  `yellow_count_prev3` which was anti-predictive: counted resolved yellows equally).
- `laps_since_last_yellow` — laps since last non-green status; capped at 10.
  Lower values = more recent danger zone. Causal: uses shift(1) of the non-green flag.
- `status_change_direction` — directional severity change: +1 escalating, −1 de-escalating,
  0 unchanged or first lap. More informative than binary `status_changed`.
- `status_changed` — retained (binary: did track status change from the previous lap?).

**From `rcm_store`** — v3 additions on top of v2 fixed parsing:
- `incident_escalation` — interaction: previous lap had incident message AND status
  changed this lap. Binary. Captures the specific precursor pattern. Causal (lag-1).
- `had_incident_msg`, `yellow_sectors_this_lap`, `yellow_sectors_prev3`,
  `rcm_incident_count_prev3` — retained from v2.

**Step 3.5 (applied at export time):**
- `circuit_sc_rate` — historical SC rate per circuit cluster, causally computed by year.


In [ ]:
# ── Step 3 — Feature Engineering ──────────────────────────────────────────────

def compute_track_status_features(grp: pd.DataFrame) -> pd.DataFrame:
    """Add track-status derived features to a single-race lap table.

    v3 additions:
      yellow_escalation_count — replaces yellow_count_prev3; counts laps in prev 3
        where track_status severity INCREASED (green→yellow, yellow→SC direction).
        Anti-predictive yellow_count_prev3 counted all yellows including resolved ones.
      laps_since_last_yellow  — laps since last non-green status; capped at 10.
        Lower = more recent danger zone. 10 = no yellow yet or too far back.
      status_changed          — retained (binary: did severity change at all?)

    Removed:
      laps_under_sc — always 0 post-labeling (SC laps dropped before export).
    """
    grp = grp.copy().sort_values("LapNumber")

    # ── yellow_escalation_count (replaces yellow_count_prev3) ────────────────
    # severity of each lap's track status
    sev = grp["track_status"].map(STATUS_SEVERITY).fillna(1).astype(int)
    # escalation: this lap's severity is HIGHER than the previous lap's
    escalated = (sev > sev.shift(1).fillna(1)).astype(int)
    # count of escalating laps in the previous 3 (shift 1 = causal)
    grp["yellow_escalation_count"] = (
        escalated.shift(1).rolling(3, min_periods=1).sum().fillna(0).astype(int)
    )

    # ── laps_since_last_yellow ────────────────────────────────────────────────
    # A lap is "yellow or worse" if track_status != '1'
    is_nongr = (grp["track_status"] != "1").astype(int)
    lsl = []
    since = 10  # default: no recent yellow
    for val in is_nongr:
        if val:
            since = 0
        else:
            since = min(since + 1, 10)
        lsl.append(since)
    # shift by 1 so current lap doesn't use its own status (causal)
    grp["laps_since_last_yellow"] = pd.Series(lsl, index=grp.index).shift(1).fillna(10).clip(upper=10).astype(int)

    # ── status_changed (binary, retained) ────────────────────────────────────
    grp["status_changed"] = (
        (grp["track_status"] != grp["track_status"].shift(1)).astype(int)
    )
    grp.loc[grp["LapNumber"] == grp["LapNumber"].min(), "status_changed"] = 0

    return grp


def build_clean_incident_mask(rcm_lap: pd.DataFrame) -> pd.Series:
    """Boolean mask for genuine on-track incident precursor events.

    Key changes vs v1:
    - SAFETY CAR messages removed — these are the SC deployment itself, not precursors.
      Including them caused leakage (the SC message fires on SC laps, which correlate
      with the label but were silently removed by the ±1 lap window trick).
    - Category == 'CarEvent' removed — too broad; includes administrative car events
      (penalties, pit lane speed, grid changes) that are not incident precursors.
    - Scope filter added: only Track/Sector scope events are on-track signals.
    """
    caution_flag = rcm_lap["Flag"].isin(["YELLOW", "DOUBLE YELLOW", "RED"])

    INCIDENT_RE = r"INCIDENT|COLLISION|CONTACT|SPIN|OFF TRACK|STOPPED CAR|DEBRIS|MARSHAL"
    EXCLUDE_RE  = r"TRACK LIMITS|LAP TIME|PENALTY|PIT LANE|FORMATION|GRID|DRS|SAFETY CAR|VIRTUAL"
    keyword_match = (
        rcm_lap["Message"].str.upper().str.contains(INCIDENT_RE, na=False, regex=True) &
        ~rcm_lap["Message"].str.upper().str.contains(EXCLUDE_RE, na=False, regex=True)
    )

    if "Scope" in rcm_lap.columns:
        track_scope = rcm_lap["Scope"].str.upper().isin(["TRACK", "SECTOR"]) | rcm_lap["Scope"].isna()
    else:
        track_scope = pd.Series(True, index=rcm_lap.index)

    return (caution_flag | keyword_match) & track_scope


def compute_rcm_features(grp: pd.DataFrame, rcm: pd.DataFrame) -> pd.DataFrame:
    """Add race-control-message features with fixed parsing.

    v3 additions:
      incident_escalation — binary: had_incident_msg (previous lap) × status_changed
        (this lap). Captures the pattern "incident last lap AND status changed" which
        is a strong SC precursor. Uses LAGGED had_incident_msg (causal).

    Retained features:
      had_incident_msg, yellow_sectors_this_lap, yellow_sectors_prev3,
      rcm_incident_count_prev3.

    Changes vs v1:
    - flags_escalating removed entirely (forward leakage).
    - Clean incident mask applied (see build_clean_incident_mask).
    - ±1 lap window in join handles RCM reporting latency (off-by-one bug).
    """
    grp = grp.copy()
    empty_cols = {
        "had_incident_msg": 0, "yellow_sectors_this_lap": 0,
        "yellow_sectors_prev3": 0, "rcm_incident_count_prev3": 0,
        "incident_escalation": 0,
    }
    if rcm.empty or "Lap" not in rcm.columns:
        for k, v in empty_cols.items():
            grp[k] = v
        return grp

    rcm_lap = rcm.dropna(subset=["Lap"]).copy()
    rcm_lap["Lap"] = rcm_lap["Lap"].astype(int)
    valid_laps = set(grp["LapNumber"].astype(int))

    # ── had_incident_msg (clean mask + ±1 lap window) ────────────────────────
    clean_mask = build_clean_incident_mask(rcm_lap)
    incident_laps_raw = set(rcm_lap[clean_mask]["Lap"])
    incident_laps = {
        l for raw_l in incident_laps_raw for l in (raw_l - 1, raw_l, raw_l + 1)
    } & valid_laps
    grp["had_incident_msg"] = grp["LapNumber"].astype(int).isin(incident_laps).astype(int)

    # ── rcm_incident_count_prev3 (rolling count of clean incidents) ───────────
    incident_per_lap = rcm_lap[clean_mask].groupby("Lap").size()
    grp["_inc"] = grp["LapNumber"].astype(int).map(incident_per_lap).fillna(0)
    grp["rcm_incident_count_prev3"] = (
        grp["_inc"].shift(1).rolling(3, min_periods=1).sum().fillna(0).astype(int)
    )
    grp = grp.drop(columns=["_inc"])

    # ── yellow_sectors_this_lap + yellow_sectors_prev3 ────────────────────────
    if "Scope" in rcm_lap.columns:
        sect_yellow = (
            rcm_lap["Scope"].str.upper().str.contains("SECTOR", na=False) &
            (rcm_lap["Flag"].str.upper().str.contains("YELLOW", na=False) |
             rcm_lap["Message"].str.upper().str.contains("YELLOW", na=False))
        )
    else:
        sect_yellow = pd.Series(False, index=rcm_lap.index)

    sy_per_lap = rcm_lap[sect_yellow].groupby("Lap").size()
    grp["yellow_sectors_this_lap"] = (
        grp["LapNumber"].astype(int).map(sy_per_lap).fillna(0).astype(int)
    )
    grp["yellow_sectors_prev3"] = (
        grp["yellow_sectors_this_lap"].shift(1).rolling(3, min_periods=1).sum().fillna(0).astype(int)
    )

    # ── incident_escalation (lagged had_incident_msg × status_changed) ────────
    # Uses the PREVIOUS lap's had_incident_msg (shift(1)) to stay causal.
    # "An incident was reported last lap AND track status changed this lap" is a
    # reliable pattern preceding SC deployments.
    grp["incident_escalation"] = (
        (grp["had_incident_msg"].shift(1).fillna(0).astype(int)) *
        grp["status_changed"].astype(int)
    ).astype(int)

    return grp


def engineer_features(raw: pd.DataFrame, rcm_store: dict) -> pd.DataFrame:
    """Apply track-status and RCM features to every race in raw."""
    feat_rows = []
    for race_id, grp in raw.groupby("race_id"):
        grp = grp.sort_values("LapNumber")
        grp = compute_track_status_features(grp)
        grp = compute_rcm_features(grp, rcm_store.get(race_id, pd.DataFrame()))
        feat_rows.append(grp)
    return pd.concat(feat_rows, ignore_index=True)


# ── Run ───────────────────────────────────────────────────────────────────────
df = engineer_features(raw, rcm_store)

print(f"Shape: {df.shape}")
print(f"\nDriver-level anomaly features:")
print(f"  driver_anomaly_soft_count > 0: {(df['driver_anomaly_soft_count'] > 0).sum()}")
print(f"  driver_anomaly_hard_count > 0: {(df['driver_anomaly_hard_count'] > 0).sum()}")
print(f"  max_speed_drop_pct > 0.05   : {(df['max_speed_drop_pct'] > 0.05).sum()}")
print(f"  tyre_age_high_risk > 0      : {(df['tyre_age_high_risk_count'] > 0).sum()}")
print(f"  active_pitstop_count > 0    : {(df['active_pitstop_count'] > 0).sum()}")
print(f"  track_temp mean             : {df['track_temp'].mean():.1f}°C" if df['track_temp'].notna().any() else "  track_temp: all NaN (weather unavailable)")
print(f"  weather_available = 1       : {df['weather_available'].sum()} laps ({df['weather_available'].mean()*100:.1f}%)")
print(f"\nFixed RCM features:")
print(f"  had_incident_msg = 1        : {df['had_incident_msg'].sum()} ({df['had_incident_msg'].mean()*100:.1f}%)")
print(f"  yellow_sectors_this_lap > 0 : {(df['yellow_sectors_this_lap'] > 0).sum()}")
print(f"  rcm_incident_count_prev3 > 0: {(df['rcm_incident_count_prev3'] > 0).sum()}")
print(f"  incident_escalation = 1     : {df['incident_escalation'].sum()}")
print(f"\nStatus features (v3):")
print(f"  yellow_escalation_count > 0 : {(df['yellow_escalation_count'] > 0).sum()}")
print(f"  laps_since_last_yellow < 10 : {(df['laps_since_last_yellow'] < 10).sum()}")
print(f"  status_changed = 1          : {df['status_changed'].sum()}")
print(f"  status_change_direction dist:\n{df['status_change_direction'].value_counts().sort_index()}")
print(f"\nNew pace features:")
print(f"  lap_time_cv mean            : {df['lap_time_cv'].mean():.4f}")
print(f"  lap_time_trend_5 != 1.0     : {(df['lap_time_trend_5'] != 1.0).sum()}")
print(f"  n_drivers_delta != 0        : {(df['n_drivers_delta'] != 0).sum()}")


**Step 3 results — 3,506 rows, 23 features.**

| Feature | Count | % laps | Notes |
|---|---|---|---|
| `yellow_count_prev3 > 0` | 264 | 7.5% | laps preceded by at least one yellow |
| `laps_under_sc > 0` | 173 | 4.9% | matches Step 2 SC count exactly ✅ |
| `status_changed = 1` | 211 | 6.0% | lap-to-lap status transitions |
| `had_incident_msg = 1` | 660 | 18.8% | yellow flags, incidents, car events, SC messages |
| `yellow_sectors_prev3 > 0` | 417 | 11.9% | sector yellows in 3-lap window |
| `flags_escalating = 1` | 105 | 3.0% | EDA only — excluded from model |

18.8% of laps carry at least one incident signal — plausible for F1, roughly one
notable on-track event every 5–6 laps. The 105 `flags_escalating` laps confirm the
signal has real predictive content: an incident message followed by SC/VSC within
one lap in ~3% of all racing laps.


> **Note for the Multi-Agent System (v0.10.0):** `session.race_control_messages`
> surfaces the full stream of FIA timing data — incident reports with car numbers,
> sector yellow flags, track limit violations, penalty announcements, safety car
> deployments, and DRS enable/disable events. This is essentially the same
> information a race engineer monitors on their timing screen.
>
> In the multi-agent architecture, a dedicated **Race Control Monitor** sub-agent
> could subscribe to this stream in real-time and feed structured signals to the
> Strategy Agent — complementing the Radio Agent (driver comms) with official FIA
> messages. The two sources together (radio sentiment + race control facts) give
> the Strategy Agent a much richer picture of the current race situation than
> either source alone.


## Step 3.5 — Circuit Historical SC Rate

One important contextual signal is the **historical SC deployment rate per circuit cluster**,
computed causally: for a race in year Y, we use only data from years < Y.

- **2023 races** — no prior data available → assign a global prior of **0.15**
  (roughly the overall observed positive rate before this dataset was built).
- **2024 races** — use 2023 rates per cluster from the races already loaded.
- **2025 races** — use 2023 + 2024 rates per cluster.

This feature is computed at the dataset level after all races are loaded, using
the labeled dataset (so it must be applied after Step 4). Because it is entirely
derived from past-season data, there is no temporal leakage.

`circuit_sc_rate` will carry the same value for every lap of a given race
(it is a circuit-level constant, not a lap-level signal). Its role is to provide
a prior that helps the model distinguish high-risk circuits (street circuits, cluster 3)
from low-risk ones (power circuits, cluster 0).


In [ ]:
# ── Step 3.5 — Circuit historical SC rate (causal) ────────────────────────────
# NOTE: This function is called AFTER Step 4 labeling, using the labeled DataFrame.
# It is defined here for clarity but executed in Step 4 / Step 6.

SC_PRIOR = 0.15  # prior for 2023 (no historical data)


def compute_circuit_sc_rate(labeled: pd.DataFrame) -> pd.DataFrame:
    """
    Add `circuit_sc_rate` to the labeled dataset using only past-season data.

    Computation:
      For each race in year Y and cluster C:
        - 2023 → SC_PRIOR (no prior data)
        - 2024 → rate of sc_within_5_laps=1 in cluster C across 2023 races
        - 2025 → rate of sc_within_5_laps=1 in cluster C across 2023+2024 races
      If a cluster has no prior laps (e.g. new circuit category), fall back to
      SC_PRIOR.

    The rate is de-duplicated at the race-lap level before aggregation (one row
    per race rather than ~60 rows per race) to avoid inflating the count.
    """
    df = labeled.copy()
    df["circuit_sc_rate"] = SC_PRIOR  # default

    # One representative row per race for rate computation
    race_level = (
        df.drop_duplicates(subset=["race_id"])
        [["race_id", "year", "circuit_cluster", "sc_within_5_laps"]]
        .copy()
    )

    SEASONS_ORDERED = sorted(df["year"].unique())

    for yr in SEASONS_ORDERED:
        if yr == min(SEASONS_ORDERED):
            # No prior data — use SC_PRIOR for all clusters
            df.loc[df["year"] == yr, "circuit_sc_rate"] = SC_PRIOR
            continue

        # Past years' race-level SC rate per cluster
        past = race_level[race_level["year"] < yr]
        if past.empty:
            df.loc[df["year"] == yr, "circuit_sc_rate"] = SC_PRIOR
            continue

        cluster_rate = (
            past.groupby("circuit_cluster")["sc_within_5_laps"]
            .mean()
            .to_dict()
        )

        # Map to current year's races by cluster
        for race_id, grp_idx in df[df["year"] == yr].groupby("race_id").groups.items():
            clust = df.loc[grp_idx[0], "circuit_cluster"]
            rate  = cluster_rate.get(clust, SC_PRIOR)
            df.loc[grp_idx, "circuit_sc_rate"] = round(float(rate), 4)

    return df


# ── Preview — will be applied after Step 4 labeling ───────────────────────────
print("Step 3.5: compute_circuit_sc_rate() defined.")
print(f"SC prior for 2023 races: {SC_PRIOR}")
print("Function will be called after Step 4 labeling — see Step 6 export cell.")


---

## Step 4 — Labeling

We define two binary targets:

- **`sc_within_5_laps`** — 1 if SC/VSC deploys within the next 5 laps (primary label,
  used in the Strategy Agent window simulation).
- **`sc_within_3_laps`** — 1 if SC/VSC deploys within the next 3 laps (tighter window;
  fewer but cleaner positives — the 5-lap look-ahead includes laps where the race state
  carries essentially no signal, diluting the positive class).

Both windows are computed simultaneously. The 5-lap horizon corresponds to the window in
which a pit stop decision can be influenced by an anticipated SC. The 3-lap variant is
included for N14 comparison: tighter windows often yield better AUC-PR because true
precursor signals (driver anomaly, yellow flag escalation) are only observable 1–3 laps
before the deployment, not 4–5.

Laps already under SC/VSC (`track_status ∈ {'4', '6'}`) are labeled -1 and **dropped**
before export — the model is never queried mid-SC period.


In [ ]:
# ── Step 4 — Labeling ─────────────────────────────────────────────────────────
WINDOW   = 5        # retained for backward-compat; sc_within_7_laps added as wider window
SC_CODES = {"4", "6"}


def label_single_race(grp: pd.DataFrame, window: int, sc_codes: set) -> np.ndarray:
    """Assign sc_within_N_laps label to a single race's lap table.
    Returns -1 for laps already under SC/VSC, 0/1 for others."""
    n        = len(grp)
    statuses = grp["track_status"].astype(str).tolist()
    labels   = np.zeros(n, dtype=int)
    for i, s in enumerate(statuses):
        if s in sc_codes:
            labels[i] = -1
            continue
        future    = statuses[i + 1 : i + window + 1]
        labels[i] = int(any(s in sc_codes for s in future))
    return labels


def build_labeled_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Apply labeling for 7-lap, 5-lap and 3-lap windows. Drop already-under-SC laps."""
    labeled_rows = []
    for _, grp in df.groupby("race_id"):
        grp = grp.copy().sort_values("LapNumber").reset_index(drop=True)
        grp["sc_within_7_laps"] = label_single_race(grp, window=7, sc_codes=SC_CODES)
        grp["sc_within_5_laps"] = label_single_race(grp, window=5, sc_codes=SC_CODES)
        grp["sc_within_3_laps"] = label_single_race(grp, window=3, sc_codes=SC_CODES)
        labeled_rows.append(grp)

    labeled    = pd.concat(labeled_rows, ignore_index=True)
    already_sc = (labeled["sc_within_5_laps"] == -1).sum()
    labeled    = labeled[
        (labeled["sc_within_7_laps"] != -1) &
        (labeled["sc_within_5_laps"] != -1) &
        (labeled["sc_within_3_laps"] != -1)
    ].reset_index(drop=True)

    pos7  = (labeled["sc_within_7_laps"] == 1).sum()
    pos5  = (labeled["sc_within_5_laps"] == 1).sum()
    pos3  = (labeled["sc_within_3_laps"] == 1).sum()
    total = len(labeled)

    print(f"Total labeled rows      : {total}")
    print(f"Already-SC dropped      : {already_sc}")
    print(f"Positive (7-lap window) : {pos7} ({pos7/total*100:.1f}%)")
    print(f"Positive (5-lap window) : {pos5} ({pos5/total*100:.1f}%)")
    print(f"Positive (3-lap window) : {pos3} ({pos3/total*100:.1f}%)")
    print(f"\nLabel by year (7-lap):")
    print(
        labeled.groupby("year")["sc_within_7_laps"]
        .agg(["sum", "mean"])
        .rename(columns={"sum": "positives", "mean": "rate"})
        .assign(rate=lambda x: x["rate"].map("{:.1%}".format))
    )
    return labeled


# ── Run ───────────────────────────────────────────────────────────────────────
labeled = build_labeled_dataset(df)

**Step 4 results — 3,275 labeled rows, 5.6% positive rate.**

| | Count | % |
|---|---|---|
| Positive `sc_within_5_laps = 1` | 183 | 5.6% |
| Negative `sc_within_5_laps = 0` | 3,092 | 94.4% |
| Dropped (already under SC/VSC) | 231 | — |

The 5.6% positive rate is lower than the naive estimate from the raw caution rate
(6.6%) because SC and VSC periods span multiple consecutive laps — a typical SC
deployment lasts 3–8 laps, all of which are dropped as already-under-caution.
Only the ~5 laps *before* each deployment are tagged as positive. Across 58 races
there are roughly 30–35 distinct SC/VSC deployment events, yielding ~183 positive
laps total.

**Class imbalance: ~18:1** — more severe than the overtake dataset (~11:1).
N14 will need `class_weight='balanced'` or equivalent to handle this.

**Year breakdown:**
- 2023: 6.8% — typical season, several high-incident circuits
- 2024: 3.0% — notably clean season, fewer SC deployments
- 2025: 6.8% — back to normal incidence rate

The 2024 anomaly is worth keeping in mind for temporal validation in N14:
the train set (2023+2024) mixes two very different SC-rate seasons, which
may make the 2024 contribution under-representative of typical race conditions.


## Step 5 — EDA

Exploratory analysis of the labeled dataset across four angles:

1. **Class distribution** — overall imbalance and breakdown by year.
2. **Positive rate by circuit cluster** — whether circuit type correlates with
   SC probability; street circuits (cluster 3) are expected to show higher rates.
3. **Positive rate by race phase** — SC deployments are not uniformly distributed;
   the opening laps (high tyre deg, cold tyres) and mid-race tend to see more incidents.
4. **Feature correlation with the label** — point-biserial correlation for numeric
   features and mean-by-label for binary flags; identifies which engineered features
   carry the most predictive signal before N14.


In [ ]:
# ── Step 5 — EDA plot functions ───────────────────────────────────────────────
COLORS_SC = {"pos": "#e74c3c", "neg": "#3498db", "neutral": "#7f8c8d"}


def plot_class_distribution(df: pd.DataFrame, save_path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    counts = df["sc_within_5_laps"].value_counts().sort_index()
    axes[0].bar(["Negative (0)", "Positive (1)"], counts.values,
                color=[COLORS_SC["neg"], COLORS_SC["pos"]])
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + 15, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=10)
    axes[0].set_title("Global class balance")
    axes[0].set_ylabel("Lap count")

    rates = df.groupby("year")["sc_within_5_laps"].mean() * 100
    axes[1].bar(rates.index.astype(str), rates.values, color=COLORS_SC["pos"], alpha=0.85)
    for i, (yr, v) in enumerate(rates.items()):
        axes[1].text(i, v + 0.1, f"{v:.1f}%", ha="center", fontsize=10)
    axes[1].set_title("SC positive rate by year")
    axes[1].set_ylabel("SC within 5 laps (%)")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path}")


def plot_sc_rate_by_cluster(df: pd.DataFrame, save_path) -> None:
    cluster_names = {0: "Power (0)", 1: "High-spd (1)", 2: "Balanced (2)", 3: "High-df (3)"}
    cluster_stats = (
        df.groupby("circuit_cluster")["sc_within_5_laps"]
        .agg(rate="mean", count="count").reset_index()
    )
    overall = df["sc_within_5_laps"].mean() * 100
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(cluster_stats["circuit_cluster"].map(cluster_names),
                  cluster_stats["rate"] * 100, color=COLORS_SC["pos"], alpha=0.85)
    ax.axhline(overall, color="gray", linestyle="--", linewidth=1, label=f"Overall {overall:.1f}%")
    for bar, (_, row) in zip(bars, cluster_stats.iterrows()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                f"n={int(row['count'])}", ha="center", fontsize=9)
    ax.set_title("SC positive rate by circuit cluster")
    ax.set_ylabel("SC within 5 laps (%)")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path}")


def plot_sc_rate_by_phase(df: pd.DataFrame, save_path) -> None:
    tmp = df.copy()
    tmp["phase_bin"] = pd.cut(tmp["lap_pct"], bins=10,
                              labels=[f"{i*10}–{(i+1)*10}%" for i in range(10)])
    phase_rate = tmp.groupby("phase_bin", observed=True)["sc_within_5_laps"].mean() * 100
    overall = df["sc_within_5_laps"].mean() * 100
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(len(phase_rate)), phase_rate.values, color=COLORS_SC["neutral"], alpha=0.85)
    ax.axhline(overall, color="gray", linestyle="--", linewidth=1, label=f"Overall {overall:.1f}%")
    ax.set_xticks(range(len(phase_rate)))
    ax.set_xticklabels(phase_rate.index, rotation=40, ha="right", fontsize=9)
    ax.set_title("SC positive rate by race phase")
    ax.set_ylabel("SC within 5 laps (%)")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path}")


def plot_feature_correlations(df: pd.DataFrame, save_path) -> None:
    """Point-biserial correlation of model features with sc_within_5_laps."""
    features = [
        # Lap pace
        "lap_time_std", "lap_time_mean", "lap_time_min",
        # Field state
        "tyre_life_mean", "lap_pct", "n_drivers",
        # Driver-level anomaly (new)
        "driver_anomaly_count", "max_speed_drop_pct",
        "tyre_age_high_risk_count", "active_pitstop_count", "outlap_drivers",
        # Track status
        "status_changed", "yellow_count_prev3",
        # RCM (fixed)
        "had_incident_msg", "yellow_sectors_this_lap",
        "yellow_sectors_prev3", "rcm_incident_count_prev3",
        # Weather
        "track_temp", "track_temp_delta", "humidity",
    ]
    # Only include features present in df
    features = [f for f in features if f in df.columns]

    corrs = pd.Series({
        f: stats.pointbiserialr(df["sc_within_5_laps"], df[f].fillna(0))[0]
        for f in features
    }).sort_values()

    colors = [COLORS_SC["pos"] if v > 0 else COLORS_SC["neg"] for v in corrs.values]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(corrs.index, corrs.values, color=colors, alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Point-biserial correlation with sc_within_5_laps")
    ax.set_title("Feature correlation vs SC label (v2 features)")
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path}")
    print("\nCorrelations (sorted descending):")
    print(corrs.sort_values(ascending=False).round(4).to_string())

In [ ]:
# ── Step 5 — Run EDA plots ────────────────────────────────────────────────────
plot_class_distribution(labeled,   OUTPUTS / "sc_class_distribution.png")
plot_sc_rate_by_cluster(labeled,   OUTPUTS / "sc_rate_by_cluster.png")
plot_sc_rate_by_phase(labeled,     OUTPUTS / "sc_rate_by_phase.png")
plot_feature_correlations(labeled, OUTPUTS / "sc_feature_correlations.png")


**Step 5 results — EDA complete, 4 plots saved.**

### Class distribution

Confirmed **18:1 imbalance** (3,092 neg / 183 pos). The inter-season pattern holds:
2024 was anomalously clean (3.0%) against 2023 and 2025 (both 6.8%). This matters for
N14 — the training set will mix two very different SC-rate regimes.

### SC rate by circuit cluster

The most surprising finding of the EDA:

| Cluster | Rate | n |
|---------|------|---|
| Balanced (2) | **14.5%** | 576 |
| Power (0) | 5.0% | 727 |
| High-spd (1) | 4.0% | 1,015 |
| High-df (3) | 2.5% | 957 |

Cluster 2 ("balanced") is 2.6× the overall average. Counter-intuitive — street circuits
(cluster 3) were expected to lead. Mixed-characteristic circuits (Spa, Suzuka, Australia)
appear to accumulate more incidents. `circuit_cluster` will be a relevant feature in N14.

### SC rate by race phase

Clear bimodal pattern:
- **Peak at 20–30%**: ~10% — peak traffic phase post-lap 1, cold tyres, position battles
- **Secondary peak at 60–70% and 80–90%**: ~6.8–7.2% — second incident window as
  retirements accumulate and strategy gaps diverge
- **30–60%**: minimum (~3.5–4%), settled racing with established gaps

### Feature correlations with the label

All correlations are weak (max 0.064) — expected given strong imbalance and dispersed signal:

- `status_changed` (0.064) — strongest linear predictor; a lap where track status
  just changed is the most direct observable precursor of an SC deployment
- `lap_time_std` (0.026) and `yellow_sectors_prev3` (0.017) — field spread and sector
  yellows carry a small but consistent positive signal
- `had_incident_msg` (-0.003) — near zero. May still contribute via non-linear
  interactions in LightGBM, but has no standalone linear predictive power
- `retirements`, `rainfall`, `laps_under_sc` → NaN — near-zero variance; worth
  investigating before including in N14

> Weak correlations are not a blocker: LightGBM captures feature interactions
> that point-biserial misses entirely. The real signal likely lives in combinations
> such as `status_changed=1 AND yellow_sectors_prev3>0 AND lap_pct<0.3`.


---

## Step 6 — Export

Export the labeled dataset to Parquet for use in N14. A JSON sidecar records
key labeling statistics and the model feature list for reproducibility.

**Feature set v3** — 35 features. Key changes from v2:

**Fixed bugs:**
- `driver_anomaly_count` replaced by dual-threshold split: `driver_anomaly_soft_count`
  (>1.15× rolling median, picks up slow drivers) and `driver_anomaly_hard_count`
  (>1.30×, crash-level spikes). The v2 single threshold of 1.5× yielded only 7
  non-zero rows across the entire dataset — effectively a useless feature.
- `yellow_count_prev3` replaced by `yellow_escalation_count` — counts only laps
  where track status severity *increased* in the previous 3 laps. The original
  feature was anti-predictive (corr = −0.012) because it counted resolved yellows
  equally with fresh ones.
- Weather features fixed: `ffill().bfill()` within each session before global
  median fallback, eliminating the constant-fill bug. `weather_available` flag
  added to let the model down-weight weather signals when sensor data is absent.

**New features:**
- `laps_since_last_yellow` — temporal memory: laps since last non-green status,
  capped at 10. Lower = more recent danger. Captures the "danger window" concept.
- `incident_escalation` — interaction binary: previous lap had incident message
  AND track status changed this lap. A strong SC precursor pattern.
- `status_change_direction` — directional change: +1 escalating, −1 de-escalating,
  0 unchanged. More informative than the binary `status_changed`.
- `lap_time_cv` — coefficient of variation (std/mean); normalises pace spread
  across circuits with very different nominal lap times.
- `lap_time_trend_5` — ratio of last-5 to prev-5 lap time means; captures pace
  deterioration within a stint (e.g. tyres going off → more variance → higher SC risk).
- `n_drivers_delta` — lap-to-lap change in active driver count; negative values
  signal a retirement just happened (car stopped on track = SC risk).
- `circuit_sc_rate` — historical SC rate per circuit cluster, computed causally
  by year (2023 = prior 0.15; 2024 = 2023 rates; 2025 = 2023+2024 rates).
  Gives the model a base-rate prior for each circuit archetype.
- `weather_available` — boolean: 1 if weather sensor data had non-trivial variance.

**Removed from v2:** `driver_anomaly_count` (replaced), `yellow_count_prev3` (replaced).

**Two labels exported:** `sc_within_5_laps` (primary) + `sc_within_3_laps` (tighter window).

**Output:** `data/processed/sc_labeled/sc_labeled_2023_2025.parquet`


In [ ]:
# ── Step 6 — Export ────────────────────────────────────────────────────────────
EXPORT_FILE = PROCESSED / "sc_labeled_2023_2025.parquet"
STATS_FILE  = PROCESSED / "labeling_stats.json"

# ── Apply Step 3.5: circuit historical SC rate (causal) ───────────────────────
# Must run AFTER Step 4 labeling so sc_within_5_laps exists.
labeled = compute_circuit_sc_rate(labeled)
print(f"circuit_sc_rate added: {labeled['circuit_sc_rate'].describe().round(4).to_dict()}")

# Model features v3 — full feature engineering improvement pass
MODEL_FEATURES = [
    # Lap pace
    "lap_time_mean", "lap_time_std", "lap_time_min",
    "lap_time_cv", "lap_time_trend_5",
    # Field state
    "n_drivers", "n_drivers_delta",
    "tyre_life_mean", "tyre_life_max",
    "tyre_age_high_risk_count", "active_pitstop_count", "outlap_drivers",
    # Driver-level anomaly signals (dual threshold)
    "driver_anomaly_soft_count", "driver_anomaly_hard_count", "max_speed_drop_pct",
    # Track status (v3)
    "track_status_enc", "status_changed", "status_change_direction",
    "yellow_escalation_count", "laps_since_last_yellow",
    # Race control messages (clean parsing)
    "had_incident_msg", "incident_escalation",
    "yellow_sectors_this_lap", "yellow_sectors_prev3", "rcm_incident_count_prev3",
    # Weather (from session.weather, not session.laps)
    "track_temp", "air_temp", "humidity", "track_temp_delta", "weather_available",
    # Race context
    "circuit_cluster", "circuit_sc_rate", "lap_pct", "is_lap1",
]


def export_labeled_dataset(labeled: pd.DataFrame, export_file, stats_file) -> None:
    labeled.to_parquet(export_file, index=False)

    pos7  = int((labeled["sc_within_7_laps"] == 1).sum())
    pos5  = int((labeled["sc_within_5_laps"] == 1).sum())
    pos3  = int((labeled["sc_within_3_laps"] == 1).sum())
    neg   = int((labeled["sc_within_7_laps"] == 0).sum())
    total = len(labeled)

    stats = {
        "export_file"        : str(export_file),
        "total_rows"         : total,
        "positive_rows_7lap" : pos7,
        "positive_rows_5lap" : pos5,
        "positive_rows_3lap" : pos3,
        "negative_rows"      : neg,
        "positive_rate_7lap" : round(pos7 / total, 4),
        "positive_rate_5lap" : round(pos5 / total, 4),
        "positive_rate_3lap" : round(pos3 / total, 4),
        "seasons"            : sorted(labeled["year"].unique().tolist()),
        "races"              : int(labeled["race_id"].nunique()),
        "label_windows"      : [3, 5, 7],
        "sc_codes"           : sorted(SC_CODES),
        "model_features"     : MODEL_FEATURES,
        "feature_version"    : "v3",
    }

    with open(stats_file, "w") as f:
        json.dump(stats, f, indent=2)

    print(f"Saved parquet → {export_file}")
    print(f"Saved stats   → {stats_file}")
    print(f"\nRows                : {total:,}")
    print(f"Positive (7-lap)    : {pos7} ({pos7/total*100:.1f}%)")
    print(f"Positive (5-lap)    : {pos5} ({pos5/total*100:.1f}%)")
    print(f"Positive (3-lap)    : {pos3} ({pos3/total*100:.1f}%)")
    print(f"Seasons             : {stats['seasons']}")
    print(f"Model features (v3) : {len(MODEL_FEATURES)}")
    missing = [f for f in MODEL_FEATURES if f not in labeled.columns]
    if missing:
        print(f"[WARN] Missing features: {missing}")
    else:
        print(f"All {len(MODEL_FEATURES)} features present in export ✓")


# ── Run ────────────────────────────────────────────────────────────────────────
export_labeled_dataset(labeled, EXPORT_FILE, STATS_FILE)

Labeled dataset saved to `data/processed/sc_labeled/sc_labeled_2023_2025.parquet`
— 3,275 rows, 15 model features, 5.6% positive rate across 58 races (2023–2025).

---

## Summary

N13 builds the labeled dataset for the Safety Car probability model.

Starting from FastF1 lap data, track status changes, and race control messages
across **58 races (2023–2025)**, the notebook produces a race-lap level dataset
with 15 engineered features and a binary label: `sc_within_5_laps`.

**Key numbers:**
- 3,275 labeled laps (231 dropped — already under SC/VSC)
- 5.6% positive rate (~18:1 class imbalance)
- 15 model features covering lap pace, tyre state, track status history, and incident signals

**Main EDA findings:**
- Balanced circuits (cluster 2) show 14.5% SC rate — 2.6× the overall average
- SC deployments concentrate in the first 30% of the race and resurge at 60–90%
- `status_changed` and `lap_time_std` are the strongest linear predictors; most
  signal is non-linear and will be captured by the tree-based model in N14
- 2024 was anomalously clean (3.0% vs 6.8% for 2023 and 2025)

**Exports:**
- `data/processed/sc_labeled/sc_labeled_2023_2025.parquet`
- `data/processed/sc_labeled/labeling_stats.json`

---

**Next → N14:** LightGBM classifier with Platt calibration, trained on 2023+2024
and tested on 2025. Output: `P(SC within N laps)` for use in the Strategy Agent's
window simulation.
